# Lab: El Niño and Fourier

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/evanwellmeyer/teaching/blob/main/ml_play/enso_fourier.ipynb)

A Fourier transform breaks a time series into waves. Here you use it on 150 years of El Niño and La Niña.

A CPU runtime is enough. Run the setup cell first.

In [ ]:
#@title Setup: imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

---
## A time series and its frequencies

Sea surface temperature in the central equatorial Pacific, the Niño 3.4 region (5°S–5°N, 170°W–120°W), is the standard measure of El Niño and La Niña. NOAA keeps a monthly record of it back to 1870 (from the HadISST dataset), as a plain text file: one row per year, one column per month.

In [ ]:
URL = "https://psl.noaa.gov/data/timeseries/month/data/nino34.long.data"
table = pd.read_csv(URL, sep=r"\s+", skiprows=1, header=None, index_col=0, nrows=2025 - 1870 + 1)   # years 1870 to 2025
values = table.values.ravel()                                        # one value per month, in order
time = pd.date_range("1870-01", periods=len(values), freq="MS")
nino34 = xr.DataArray(values, coords={"time": time})[values > -99]   # -99.99 means missing

plt.figure(figsize=(12, 3.5))
plt.plot(nino34.time, nino34, color="black", linewidth=0.8)
plt.ylabel("°C")
plt.title("Sea surface temperature in the Niño 3.4 region")
plt.show()

A Fourier transform rewrites a time series as a sum of sine waves. Wave 1 fits exactly one cycle into the whole record, wave 2 fits two cycles, wave 3 three, and so on, up to one cycle every two months. The transform gives the size and timing of each wave so that, added up (plus the mean), they rebuild the series exactly.

Here are the first five waves, and what the first 5, 10 and 20 add up to.

In [ ]:
# The first five waves, all drawn with the same size: wave k fits k cycles into the record
fraction = np.arange(len(nino34)) / len(nino34)   # 0 at the start of the record, 1 at the end
years = len(nino34) / 12

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 9))

for k in range(1, 11):
    amp = np.sin(2 * np.pi * k * fraction) - 3 * k
    pos = -3 * k

    ax1.plot(nino34.time, amp, color="black")   # wave 1 at the top, going down
    ax1.text(nino34.time[-1], pos, f"  wave {k}: one cycle every {years / k:.0f} years", va="center")

ax1.set_yticks([])
ax1.set_title("The first five waves")

# The Fourier transform gives each wave's actual size and timing. Keeping only the first n and
# transforming back adds them up (coefficient 0 is the mean).

coefficients = np.fft.rfft(nino34.values)
ax2.plot(nino34.time, nino34, color="0.75", linewidth=0.8, label="data")

for n in [5, 10, 20, 50]:
    ax2.plot(nino34.time, np.fft.irfft(coefficients[:n + 1], n=len(nino34)), linewidth=1.5, label=f"first {n} waves")

ax2.set_ylabel("°C")
ax2.set_title("Adding them up")
ax2.legend(loc="upper left")
plt.show()

Each wave has a power, its amplitude squared, which says how much of the series it carries. Plotting the power against the period of each wave (one over its frequency) gives the spectrum.

In [ ]:
values = nino34.values - nino34.values.mean()
power = np.abs(np.fft.rfft(values)) ** 2            # how strong each wave is
frequency = np.fft.rfftfreq(len(values), d=1 / 12)  # cycles per year (one value per month)
period = 1 / frequency[1:]                          # years per cycle (frequency 0 is just the mean)

ticks = [0.25, 0.5, 1, 2, 4, 8, 16, 32, 64]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(period, power[1:], color="black")
ax.set_xscale("log")
ax.set_xticks(ticks, [f"{t:g}" for t in ticks])
ax.set_xlabel("period (years)")
ax.set_ylabel("power")
ax.set_title("Fourier spectrum of Niño 3.4 sea surface temperature")
plt.show()

The tallest peak is the seasonal cycle, at one cycle per year. Remove it by subtracting the average for each calendar month (the anomaly), and take the spectrum again.

In [ ]:
nino34_anomaly = nino34.groupby("time.month") - nino34.groupby("time.month").mean()
values = nino34_anomaly.values
power = np.abs(np.fft.rfft(values)) ** 2

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7))

ax1.fill_between(nino34_anomaly.time, values, where=values > 0, color="tab:red")
ax1.fill_between(nino34_anomaly.time, values, where=values < 0, color="tab:blue")
ax1.set_ylabel("°C")
ax1.set_title("Niño 3.4 anomaly: El Niño (red) and La Niña (blue)")
ax2.plot(period, power[1:], color="black")
ax2.axvspan(2, 7, color="0.9")   # the usual range of El Niño periods
ax2.set_xscale("log")
ax2.set_xticks(ticks, [f"{t:g}" for t in ticks])
ax2.set_xlabel("period (years)")
ax2.set_ylabel("power")
ax2.set_title("Fourier spectrum of the anomaly")
plt.show()

Rebuild the anomaly from only its strongest waves. How many do you need?

In [ ]:
N_WAVES = 100

coefficients = np.fft.rfft(values)
strongest = np.argsort(np.abs(coefficients))[::-1][:N_WAVES]   # the N_WAVES largest
kept = np.zeros_like(coefficients)
kept[strongest] = coefficients[strongest]
rebuilt = np.fft.irfft(kept, n=len(values))
explained = 1 - np.var(values - rebuilt) / np.var(values)

plt.figure(figsize=(12, 3.5))
plt.plot(nino34_anomaly.time, values, color="0.6", linewidth=0.8, label="anomaly")
plt.plot(nino34_anomaly.time, rebuilt, color="black", linewidth=1.5, label=f"{N_WAVES} strongest waves")
plt.ylabel("°C")
plt.title(f"{N_WAVES} of {len(coefficients)} waves explain {explained:.0%} of the variance")
plt.legend(loc="upper left")
plt.show()

---
## Can the waves predict El Niño?

Split the anomaly in two: fit the waves on the first 100 years (the training set), then keep the same waves going into the remaining years (the test set) and see whether they match what actually happened.

In [ ]:
TRAIN_YEARS = 100   # fit on the first 100 years, test on the rest

values = nino34_anomaly.values
train = values[:12 * TRAIN_YEARS]
coefficients = np.fft.rfft(train)   # the waves of the training years only
months = np.arange(len(values))     # months since the start of the record, through the test years

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True, layout="constrained")

for ax, n in zip(axes, [10, 50, 100]):

    # Add up the first n waves, each with its size and timing from the training years, and keep going past them
    curve = np.full(len(values), coefficients[0].real / len(train))   # the mean

    for k in range(1, n + 1):
        size = 2 * np.abs(coefficients[k]) / len(train)
        curve += size * np.cos(2 * np.pi * k * months / len(train) + np.angle(coefficients[k]))

    test = slice(len(train), None)
    r = np.corrcoef(curve[test], values[test])[0, 1]
    
    ax.plot(nino34_anomaly.time, values, color="0.75", linewidth=0.8, label="anomaly")
    ax.plot(nino34_anomaly.time[:len(train)], curve[:len(train)], color="black", linewidth=1.5, label="fit (train)")
    ax.plot(nino34_anomaly.time[test], curve[test], color="tab:red", linewidth=1.5, label="continued (test)")
    ax.axvline(nino34_anomaly.time[len(train)].values, color="black", linestyle="--", linewidth=0.8)
    ax.set_ylabel("°C")
    ax.set_title(f"First {n} waves (periods down to {TRAIN_YEARS / n:g} years): correlation with the test years {r:.2f}")

axes[0].legend(loc="upper left")
plt.show()